# ЛР7 — устойчивый браузерный сбор: метрики, графики, выводы
Ноутбук читает только `logs/` и `reports/`. Сначала выполните `python run.py all`.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT / "src"))
from observability import read_log
import analysis
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
rep = lambda rid: json.loads((ROOT / "reports" / f"run_report_{rid}.json").read_text(encoding="utf-8"))

## 1. robots.txt

In [ ]:
rb = json.loads((ROOT / "state" / "robots.json").read_text(encoding="utf-8"))
pd.DataFrame([{"host": h, "HTTP": v["status"], "трактовка": v["interpretation"], "can_fetch": v["can_fetch"],
               "crawl_delay": v["crawl_delay"], "request_rate": v["request_rate"], "sitemaps": v["sitemaps"]} for h, v in rb.items()])

## 2. Прогоны A / B / C

In [ ]:
display(Markdown((ROOT / "reports" / "comparison.md").read_text(encoding="utf-8")))

In [ ]:
cols = ["run_id", "requests", "status_codes", "error_rate", "share_429", "share_403", "retries", "latency_median_ms",
        "latency_p95_ms", "bytes_wire", "not_modified", "breaker_activations", "collected", "known", "discarded", "duration_s", "status"]
pd.DataFrame([rep(r) for r in ("A", "B", "C_cold", "C_warm")])[cols]

## 3. Сценарии варианта 4 и демонстрации

In [ ]:
runs = ["S1_selector_timeout", "S2_browser_down", "S3_stub_pages", "S4_breaker_isolation", "S5_retry_policy"]
pd.DataFrame([rep(r) for r in runs])[cols + ["fallbacks", "breaker_skipped", "incidents", "detector"]]

In [ ]:
def events(rid, kinds):
    return pd.DataFrame([e for e in read_log(ROOT / "logs" / f"run_{rid}.jsonl") if e["event"] in kinds])
show = ["ts", "event", "host", "url", "status", "ms", "attempt", "error_class", "delay_s", "delay_source", "error"]
print("S1: таймаут селектора → повтор → деградация до HTTP JSON")
display(events("S1_selector_timeout", {"request", "retry", "give_up", "fallback", "incident"}).reindex(columns=show))

In [ ]:
print("S5: журнал попыток — номер, код, задержка и её источник")
display(events("S5_retry_policy", {"request", "retry", "give_up", "auth_refresh"}).reindex(columns=show))

In [ ]:
print("S4: предохранитель — переходы состояний и пропуски")
display(events("S4_breaker_isolation", {"breaker_state", "breaker_skip"}).reindex(columns=["ts", "event", "host", "old", "new", "remaining_s", "consecutive_failures"]))

In [ ]:
print("S3: детектор и инциденты")
display(events("S3_stub_pages", {"detector", "incident", "stopped_skip"}).reindex(columns=["ts", "event", "host", "url", "status", "verdict", "reasons", "artifacts"]))
print(sorted(p.name for p in (ROOT / "logs" / "incidents").iterdir()))

## 4. Оповещения по всем прогонам

In [ ]:
rows = []
for p in sorted((ROOT / "reports").glob("run_report_*.json")):
    r = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"прогон": r["run_id"], "статус": r["status"], "сработало": "; ".join(f"[{a['level']}] {a['metric']}" for a in r["alerts"]) or "—",
                 "история": r["history"].get("status")})
pd.DataFrame(rows)

## 5. Лимит при 4 параллельных процессах

In [ ]:
pd.DataFrame(json.loads((ROOT / "reports" / "parallel_limit.json").read_text(encoding="utf-8"))).T

## 6. Графики

In [ ]:
analysis.main()
for f in ("latency_hist.png", "codes_timeline_B.png", "codes_timeline_breaker.png"):
    display(Image(filename=str(ROOT / "reports" / f)))